# NB 03 — Quad Pipeline

Build the Quad reference index from `dry_ref` tracks and run all queries
from `manifest_dry.csv`. Saves results to `results/dry_run/quad_raw.csv`.

Reference: Sonnleitner & Widmer (2016), "Robust Quad-Based Audio Fingerprinting".


In [ ]:
# ── RUN MODE CONFIG ──────────────────────────────────────────────────────────
# Switch between dry run and live run by editing src/run_config.py.
import sys
from pathlib import Path
sys.path.insert(0, str(Path("..").resolve() / "src"))
from run_config import cfg
RUN_MODE = cfg.run_mode
print(f"RUN_MODE = {RUN_MODE!r}")

## 1  Setup

In [ ]:
import os, sys
from pathlib import Path

os.chdir(Path("..").resolve())
PROJECT_ROOT = Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))
print("PROJECT_ROOT:", PROJECT_ROOT)

from run_config import cfg
RUN_MODE = cfg.run_mode
print(f"RUN_MODE = {RUN_MODE!r}")

In [ ]:
import json
import logging
import time

import pandas as pd

from utils import load_fma_metadata, load_partitions
from metrics import classify_result
from quad_pipeline import build_quad_index, run_quad_query, get_true_scales

logging.basicConfig(level=logging.WARNING)


## 2  Parameters

In [ ]:
FMA_DIR      = PROJECT_ROOT / "data" / "fma_medium"
PARTS_DIR    = PROJECT_ROOT / "data" / "partitions"
MANIFEST_CSV = PROJECT_ROOT / "data" / "queries" / cfg.manifest_name
RESULTS_DIR  = PROJECT_ROOT / "results" / cfg.results_subdir
REF_KEY      = cfg.ref_key   # "dry_ref" or "live_ref"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
RAW_CSV    = RESULTS_DIR / "quad_raw.csv"
EFFIC_JSON = RESULTS_DIR / "quad_efficiency.json"

print(f"MANIFEST_CSV = {MANIFEST_CSV}")
print(f"RESULTS_DIR  = {RESULTS_DIR}")
print(f"REF_KEY      = {REF_KEY!r}")

## 3  Load partitions and manifest

In [ ]:
partitions  = load_partitions(str(PARTS_DIR))
ref_ids     = partitions[REF_KEY]
print(f"{REF_KEY}: {len(ref_ids)} tracks")

metadata_df = load_fma_metadata(str(FMA_DIR))
print(f"Metadata loaded: {len(metadata_df)} tracks")

In [ ]:
manifest = pd.read_csv(MANIFEST_CSV, dtype={"ref_track_id": "Int64"})
print(f"Manifest: {manifest.shape[0]} queries, {manifest['condition'].nunique()} conditions")
manifest.head(3)


## 4  Build Quad reference index

In [ ]:
# Tolerance parameters: epsilon_t = epsilon_p = 0.3
# (Sonnleitner & Widmer 2016 default; actual values in quad_fingerprint/config.py)
print(f"Building Quad index on {len(ref_ids)} tracks ({REF_KEY}) ...")
quad_db, build_stats = build_quad_index(ref_ids, metadata_df)
print(json.dumps(build_stats, indent=2))

## 5  Run all queries

In [ ]:
rows = []
t_wall_start = time.perf_counter()

for _, qrow in manifest.iterrows():
    track_id     = int(qrow["track_id"])
    query_path   = qrow["query_path"]
    ref_track_id = None if pd.isna(qrow["ref_track_id"]) else int(qrow["ref_track_id"])
    is_ood       = bool(qrow["is_ood"])
    condition    = qrow["condition"]

    pred_id, score, q_ms, det_t, det_f = run_quad_query(query_path, quad_db)
    result_class = classify_result(pred_id, ref_track_id, is_ood)
    true_t, true_f = get_true_scales(condition)

    rows.append({
        "system":               "quad",
        "track_id":             track_id,
        "ref_track_id":         ref_track_id,
        "is_ood":               is_ood,
        "predicted_id":         pred_id,
        "score":                score,
        "result_class":         result_class,
        "query_time_ms":        q_ms,
        "group":                qrow["group"],
        "condition":            condition,
        "duration_sec":         qrow["duration_sec"],
        "detected_time_scale":  det_t,
        "detected_freq_scale":  det_f,
        "true_time_scale":      true_t,
        "true_freq_scale":      true_f,
    })

total_wall_ms = (time.perf_counter() - t_wall_start) * 1000.0
print(f"Queries done: {len(rows)} total, {total_wall_ms/1000:.1f}s wall time")


## 6  Save results

In [ ]:
results_df = pd.DataFrame(rows)
results_df["ref_track_id"] = results_df["ref_track_id"].astype(pd.Int64Dtype())
results_df["predicted_id"] = results_df["predicted_id"].astype(pd.Int64Dtype())
results_df.to_csv(RAW_CSV, index=False)
print(f"Saved {len(results_df)} rows → {RAW_CSV}")


In [ ]:
efficiency = {
    "system":        "quad",
    "run_mode":      RUN_MODE,
    "build_stats":   build_stats,
    "total_queries": len(rows),
    "total_wall_ms": round(total_wall_ms, 3),
    "avg_query_ms":  round(total_wall_ms / len(rows), 3) if rows else 0.0,
}
with open(EFFIC_JSON, "w") as f:
    json.dump(efficiency, f, indent=2)
print(json.dumps(efficiency, indent=2))

## 7  Quick evaluation

In [ ]:
in_db = results_df[~results_df["is_ood"]]
ood   = results_df[results_df["is_ood"]]

tp = (in_db["result_class"] == "TP").sum()
fn = (in_db["result_class"] == "FN").sum()
fp_indb = (in_db["result_class"] == "FP").sum()

tn = (ood["result_class"] == "TN").sum()
fp_ood = (ood["result_class"] == "FP").sum()

hit_rate    = tp / (tp + fn + fp_indb) if (tp + fn + fp_indb) > 0 else 0.0
specificity = tn / (tn + fp_ood) if (tn + fp_ood) > 0 else 0.0

print(f"Hit Rate (In-DB):     {hit_rate:.1%}  (TP={tp}, FN={fn}, FP_indb={fp_indb})")
print(f"Specificity (OOD):    {specificity:.1%} (TN={tn}, FP_ood={fp_ood})")


In [ ]:
# Hit rate on A_original (baseline sanity check)
a_orig = results_df[results_df["condition"] == "A_original"]
a_orig_tp = (a_orig["result_class"] == "TP").sum()
a_orig_total = len(a_orig[~a_orig["is_ood"]])
print(f"Hit Rate A_original:  {a_orig_tp}/{a_orig_total} = {a_orig_tp/a_orig_total:.1%}")


## 8  Scale factor estimation (Group A)

In [ ]:
group_a = results_df[
    results_df["group"].eq("A") &
    results_df["condition"].str.match(r"A_(tempo|speed)_")
].copy()
group_a = group_a[group_a["detected_time_scale"].notna()].copy()

group_a["err_time"] = (group_a["detected_time_scale"] - group_a["true_time_scale"]).abs()
group_a["err_freq"] = (group_a["detected_freq_scale"] - group_a["true_freq_scale"]).abs()

summary = group_a.groupby("condition")[["true_time_scale","detected_time_scale","err_time"]].mean()
print(summary.to_string())


## 8b  Befund: A_tempo_* schlägt komplett fehl — Phase-Vocoder-Limitation

### Beobachtung

Alle `A_tempo_*`-Bedingungen (80 %–120 %) erzielen **0 % Hit Rate**, obwohl
`EPSILON_T = 0.31` Tempoänderungen bis ±31 % abdecken sollte.

Debugging-Trace (Track 5026, A_tempo_95):

| Metrik | A_original | A_tempo_95 |
|---|---|---|
| KD-tree-Treffer (SEARCH_RADIUS=0.01) | 53 / 4751 (1.1 %) | **1 / 5944 (0.02 %)** |
| Mediane Hash-Distanz query→ref | ~0.003 | **0.12** |

Mit nur 1 Treffer kann keine Sequenz mit `SEQUENCE_MIN_MATCHES=4` gebildet werden.

### Ursache: `librosa.effects.time_stretch` zerstört Peak-Korrespondenz bei 8 kHz

Der Quad-Hash ist theoretisch **skalierungsinvariant**:

```
C'_x = (C_t − A_t) / (B_t − A_t)
```

Skaliert man alle Zeiten um s_t, kürzt sich der Faktor heraus → Hash bleibt gleich.
Diese Invarianz gilt aber nur, wenn in Referenz **und** Query **dieselben vier Peaks**
detektiert werden.

`librosa.effects.time_stretch` (Phase-Vocoder) verteilt die Spektralenenergie bei 8 kHz
so stark um, dass der Peak-Finder andere Peaks an anderen Positionen extrahiert.
Dadurch divergieren die Query-Hashes im Mittel um **0.12** — weit jenseits von
SEARCH_RADIUS = 0.01.

### Warum funktioniert A_speed_* aber?

`apply_speed_change` verwendet `scipy.signal.resample` (reine Resampling-Operation).
Das Spektrogramm der resampleten Datei ist das **exakt skalierte** Original — Peaks
liegen an proportional verschobenen Positionen, die Hash-Komponenten kürzen sich aus.
Ergebnis: erkannte vs. wahre Skalierungsfaktoren stimmen bis auf < 0.002 überein
(siehe Tabelle oben).

### Einordnung

Dies ist eine **Limitation der Distortion-Implementierung**, nicht des Quad-Algorithmus.

Sonnleitner & Widmer (2016) verwenden für Time-Stretching **SoX** (Sound eXchange),
nicht librosa. SoX nutzt einen qualitativ hochwertigen WSOLA-basierten Algorithmus,
der die Spektralstruktur weitgehend erhält. Bei 8 kHz mit N_FFT = 1024 ist der
librosa-Phase-Vocoder für Fingerprinting-Zwecke ungeeignet.

**Wissenschaftliche Interpretation:** Der Quad-Algorithmus ist für Tempo-Robustheit
ausgelegt und funktioniert in der Originalpublikation. Die 0 % Hit Rate bei A_tempo_*
in diesem Benchmark reflektiert die Qualität der Distortion-Pipeline (librosa PV @
8 kHz), nicht eine Schwäche des Algorithmus selbst. Dieser Befund ist in der
Ergebnisdiskussion (NB 06) entsprechend zu vermerken.

## 9  Sanity check

In [ ]:
print("Results shape:", results_df.shape)
print("result_class counts:")
print(results_df["result_class"].value_counts().to_string())
print()
print("First 5 rows:")
results_df.head(5)
